# Network Anomaly Detection — Full Analysis

Complete analysis pipeline: from packet capture to anomaly detection and visualization.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
%matplotlib inline

from src.collector.packet_capture import PacketGenerator
from src.processing.feature_extractor import FeatureExtractor
from src.detection.detector import AnomalyDetector, RuleBasedDetector
from src.visualization.plots import (
    generate_all_plots, plot_time_series_anomalies,
    plot_confusion_matrix, plot_method_comparison
)
from src.comparison import run_method_comparison
from src.utils.helpers import print_detection_summary

## 1. Data Generation

Generating mixed network traffic with 4 types of attacks:

In [ ]:
gen = PacketGenerator()
attack_configs = [
    {'type': 'ddos', 'count': 300},
    {'type': 'port_scan', 'count': 200},
    {'type': 'bruteforce', 'count': 100},
    {'type': 'data_exfiltration', 'count': 150},
]
traffic_df, packet_labels = gen.generate_mixed_traffic(
    n_normal=5000, attack_configs=attack_configs
)
n_anomalies = sum(1 for v in packet_labels if v == 1)
print(f"Total packets: {len(traffic_df)}")
print(f"Normal: {len(packet_labels) - n_anomalies}")
print(f"Anomalies: {n_anomalies}")
if 'attack_type' in traffic_df.columns:
    print(traffic_df['attack_type'].value_counts())

## 2. Feature Extraction

Aggregating packets into time windows and extracting statistical features.

In [ ]:
extractor = FeatureExtractor(window_size=2.0)
window_features = extractor.aggregate_window(traffic_df)
print(f"Windows: {len(window_features)}")
print(f"Features: {list(window_features.columns)}")
window_features.head()

## 3. Prepare Ground Truth Labels

Mapping packet-level labels to window-level labels.

In [ ]:
start_time = traffic_df['timestamp'].min()
window_size = 2.0
traffic_df['window_idx'] = ((traffic_df['timestamp'] - start_time) // window_size).astype(int)
window_labels = traffic_df.groupby('window_idx')['attack_type'].apply(
    lambda x: x.notna().any()
).astype(int).values
wn = len(window_features)
window_labels = window_labels[:wn] if len(window_labels) >= wn else np.pad(
    window_labels, (0, wn - len(window_labels)), 'constant'
)
print(f"Anomalous windows: {window_labels.sum()} / {wn}")

## 4. Algorithm Comparison

Comparing Isolation Forest, LOF, and One-Class SVM.

In [ ]:
comparison_results = run_method_comparison(window_features, window_labels)
print("\nBest method:", max(comparison_results, key=lambda k: comparison_results[k]['f1_score']))

## 5. Best Method — Detailed Results

Running the best detector and combining with rule-based alerts.

In [ ]:
detector = AnomalyDetector(method='isolation_forest', contamination=0.15)
detector.fit(window_features)
full_results = detector.predict(window_features)
full_results['true_label'] = window_labels

rule_detector = RuleBasedDetector.default_rules()
rule_results = rule_detector.detect(window_features)
full_results['rule_alert'] = rule_results['has_alert']
full_results['is_alert'] = full_results['is_anomaly'] | full_results['rule_alert']

print_detection_summary(full_results)
full_results[['window', 'packet_count', 'is_anomaly', 'anomaly_score', 'true_label', 'rule_alert']].head(15)

## 6. Visualizations

Generating all plots for the report.

In [ ]:
y_true = window_labels
y_score = full_results['anomaly_score'].values
generate_all_plots(
    window_features, full_results,
    y_true=y_true, y_score=y_score,
    comparison_results=comparison_results
)
print("\nAll plots saved to data/plots/")

## 7. Display Plots Inline

In [ ]:
from IPython.display import Image, display
plot_dir = os.path.join(os.getcwd(), '..', 'data', 'plots')
for fname in ['time_series_anomalies.png', 'confusion_matrix.png',
              'anomaly_score_distribution.png', 'method_comparison.png',
              'roc_curve.png', 'feature_correlation.png']:
    path = os.path.join(plot_dir, fname)
    if os.path.exists(path):
        display(Image(filename=path))

## 8. Attack Type Analysis

Breaking down detection results by attack type.

In [ ]:
if 'window_idx' in traffic_df.columns and 'attack_type' in traffic_df.columns:
    window_attack = traffic_df.groupby('window_idx')['attack_type'].apply(
        lambda x: x.dropna().unique().tolist() if any(x.notna()) else []
    ).reset_index()
    window_attack.columns = ['window_idx', 'attack_types']
    window_attack['has_attack'] = window_attack['attack_types'].apply(len) > 0
    print("Attack distribution by window:")
    from collections import Counter
    all_attacks = []
    for types in window_attack['attack_types']:
        all_attacks.extend(types)
    for attack, count in Counter(all_attacks).most_common():
        print(f"  {attack}: {count} windows")

In [ ]:
print("\n=== ANALYSIS COMPLETE ===")
print(f"Windows analyzed: {len(full_results)}")
print(f"Anomalies detected: {full_results['is_anomaly'].sum()}")
print(f"Rule alerts: {full_results['rule_alert'].sum()}")
print(f"Combined alerts: {full_results['is_alert'].sum()}")